In [44]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import models
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import scipy.io as sio
import numpy as np
import os
import time
import wandb
from tqdm import tqdm

np.random.seed(2137)
torch.manual_seed(2137)

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Device: {device}")

Device: mps


In [26]:
labels_path = "./data/imagelabels.mat"
setid_path = "./data/setid.mat"
img_dir = "./data/jpg"

# słowniki ze zmiennymi z matlaba

labels_data = sio.loadmat(labels_path) 
setid_data = sio.loadmat(setid_path)

In [27]:
print(f"labels dict: {labels_data.keys()}")

print(f"setid dict: {setid_data.keys()}")

labels dict: dict_keys(['__header__', '__version__', '__globals__', 'labels'])
setid dict: dict_keys(['__header__', '__version__', '__globals__', 'trnid', 'valid', 'tstid'])


W labelsach jest są labelsy od 1 do 102, to bedzimy chcieli zamienic na 0 do 101, bo do loss bedzie wygodniejsze.   
A w dancyh trenigowych są też od 1 do 8189, tutaj nie ma to wiekszego znaczenia bo to jedynie ids do zdjeć. 

In [28]:
labels = labels_data['labels'].squeeze() - 1

train_ids = setid_data['trnid'].squeeze()
val_ids = setid_data['valid'].squeeze()
test_ids = setid_data['tstid'].squeeze()

print(f"Liczba zdjęć: Treningowe ({len(train_ids)}), Walidacyjne ({len(val_ids)}), Testowe ({len(test_ids)})")

Liczba zdjęć: Treningowe (1020), Walidacyjne (1020), Testowe (6149)


In [29]:
class FlowersDataset(Dataset):
    def __init__(self, ids, labels, img_dir, transform=None):
        self.ids = ids
        self.labels = labels
        self.img_dir = img_dir
        self.transform = transform
    
    def __len__(self):
        return len(self.ids)
    
    def __getitem__(self, index):
        img_id = self.ids[index]

        img_name = f"image_{img_id:05d}.jpg"
        img_path = os.path.join(self.img_dir, img_name)

        img = Image.open(img_path).convert("RGB")

        label = self.labels[img_id - 1]

        if self.transform:
            img = self.transform(img)
        
        return img, label

In [30]:
basic_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

In [31]:
train_dataset = FlowersDataset(train_ids, labels, img_dir, basic_transform)
val_dataset = FlowersDataset(val_ids, labels, img_dir, basic_transform)
test_dataset = FlowersDataset(test_ids, labels, img_dir, basic_transform)

batch_size = 32

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [32]:
print("\nPobieram próbny batch z DataLoadera...")
images, targets = next(iter(train_loader))
print(f"Kształt paczki obrazków: {images.shape} -> (Batch, Kanały, Wysokość, Szerokość)")
print(f"Kształt paczki etykiet: {targets.shape}")


Pobieram próbny batch z DataLoadera...
Kształt paczki obrazków: torch.Size([32, 3, 224, 224]) -> (Batch, Kanały, Wysokość, Szerokość)
Kształt paczki etykiet: torch.Size([32])


In [33]:
class ExampleCNN(nn.Module):
    def __init__(self, num_classes=102):
        super(ExampleCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 8, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(8, 16, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        # Fully connected layers
        self.c = 56*56*16
        self.fc1 = nn.Linear(self.c, 1024) # Flatten (512 * 7 * 7) features
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(1024, num_classes)
    def forward(self, x):
        x = self.pool1(F.relu(self.conv1(x)))
        x = self.pool2(F.relu(self.conv2(x)))
        x = x.view(-1, self.c) # Flatten the tensor
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x) # logits for all classes
        return x

In [34]:
model_cnn = ExampleCNN(num_classes=102).to(device)
print(model_cnn)

ExampleCNN(
  (conv1): Conv2d(3, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(8, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=50176, out_features=1024, bias=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc2): Linear(in_features=1024, out_features=102, bias=True)
)


In [35]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_cnn.parameters(), lr=0.001)

In [40]:
def train_model(model, train_loader, val_loader, criterion, optimizer, epochs, lr, name, architecture):
    wandb.init(
        project="flowers",
        name=name,      
        config={                      
            "learning_rate": lr,
            "architecture": architecture,
            "epochs": epochs,
            "batch_size": batch_size
        }
    )
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    start_time = time.time()

    for epoch in range(epochs):
        print(f"Epoka {epoch+1}/{epochs}")

        model.train()
        running_loss = 0.0
        running_correct = 0

        for inputs, labels in tqdm(train_loader, desc="Traning"):
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()

            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            _, preds = torch.max(outputs, 1)
            running_loss += loss.item() * inputs.size(0)
            running_correct += torch.sum(preds == labels.data)

        epoch_train_loss = running_loss / len(train_loader.dataset)
        epoch_train_acc = running_correct / len(train_loader.dataset)

        model.eval()
        val_loss = 0.0
        val_correct = 0

        with torch.no_grad():
            for inputs, labels in tqdm(val_loader, desc="Validation"):
                inputs, labels = inputs.to(device), labels.to(device)

                outputs = model(inputs)
                loss = criterion(outputs, labels)

                _, preds = torch.max(outputs, 1)
                val_loss += loss.item() * inputs.size(0)
                val_correct += torch.sum(preds == labels.data)
        
        epoch_val_loss = val_loss / len(val_loader.dataset)
        epoch_val_acc = val_correct / len(val_loader.dataset)

        wandb.log({
            "epoch": epoch + 1,
            "train_loss": epoch_train_loss,
            "train_acc": epoch_train_acc,
            "val_loss": epoch_val_loss,
            "val_acc": epoch_val_acc
        })

        history['train_loss'].append(epoch_train_loss)
        history['train_acc'].append(epoch_train_acc)
        history['val_loss'].append(epoch_val_loss)
        history['val_acc'].append(epoch_val_acc)
        
        print(f"Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.4f}")
        print(f"Val Loss:   {epoch_val_loss:.4f} | Val Acc:   {epoch_val_acc:.4f}\n")

    total_time = time.time() - start_time
    print(f"Trening zakończony w {total_time // 60:.0f}m {total_time % 60:.0f}s")
        
    wandb.log({"total_time_seconds": total_time})
    
    wandb.finish()
    
    return history

In [41]:
history_cnn = train_model(model_cnn, train_loader, val_loader, criterion, optimizer, epochs=10, lr=0.01, name="task0-cnn", architecture="example_cnn")

Epoka 1/10


Validation: 100%|██████████| 32/32 [00:03<00:00,  9.01it/s]


Train Loss: 4.4574 | Train Acc: 0.0294
Val Loss:   4.2737 | Val Acc:   0.0451

Epoka 2/10


Validation: 100%|██████████| 32/32 [00:03<00:00,  9.85it/s]


Train Loss: 4.0648 | Train Acc: 0.0569
Val Loss:   4.0369 | Val Acc:   0.0657

Epoka 3/10


...

Validation: 100%|██████████| 32/32 [00:03<00:00,  9.87it/s]


Train Loss: 0.4341 | Train Acc: 0.8961
Val Loss:   3.8710 | Val Acc:   0.2069

Epoka 10/10


Validation: 100%|██████████| 32/32 [00:03<00:00,  9.20it/s]

Train Loss: 0.2543 | Train Acc: 0.9373
Val Loss:   3.7184 | Val Acc:   0.2108

Trening zakończony w 1m 28s


epoch,▁▂▃▃▄▅▆▆▇█
total_time_seconds,▁
train_acc,▁▁▂▂▄▅▆▇██
train_loss,█▇▇▆▅▄▃▂▁▁
val_acc,▁▂▃▅▆▇▇███
val_loss,█▆▄▂▂▁▂▂▄▃
epoch,10
total_time_seconds,88.47548
train_acc,0.93725
train_loss,0.25432
val_acc,0.21078


## Treninig 10%

In [42]:
num_10_percent = len(train_ids) // 10
train_ids_10 = train_ids[:num_10_percent]

print(f"Oryginalny zbiór treningowy: {len(train_ids)} zdjęć.")
print(f"Zbiór 10%: {len(train_ids_10)} zdjęć.")

train_dataset_10 = FlowersDataset(train_ids_10, labels, img_dir, transform=basic_transform)
train_loader_10 = DataLoader(train_dataset_10, batch_size=batch_size, shuffle=True)

model_cnn_10 = ExampleCNN(num_classes=102).to(device)

optimizer_10 = optim.Adam(model_cnn_10.parameters(), lr=0.001)

print("\nRozpoczynamy trening na 10% danych...")
history_cnn_10 = train_model(
    model=model_cnn_10, 
    train_loader=train_loader_10, 
    val_loader=val_loader, 
    criterion=criterion, 
    optimizer=optimizer_10, 
    epochs=10, 
    lr=0.001, 
    name="task0-cnn-10-percent", 
    architecture="example_cnn"
)

Oryginalny zbiór treningowy: 1020 zdjęć.
Zbiór 10%: 102 zdjęć.

Rozpoczynamy trening na 10% danych...


Epoka 1/10


Validation: 100%|██████████| 32/32 [00:03<00:00,  8.27it/s]


Train Loss: 4.6891 | Train Acc: 0.0686
Val Loss:   13.2848 | Val Acc:   0.0098

Epoka 2/10


Validation: 100%|██████████| 32/32 [00:03<00:00,  9.96it/s]


...

Validation: 100%|██████████| 32/32 [00:03<00:00, 10.01it/s]


Train Loss: 1.3972 | Train Acc: 0.5490
Val Loss:   11.9892 | Val Acc:   0.0451

Epoka 10/10


Validation: 100%|██████████| 32/32 [00:03<00:00,  9.91it/s]

Train Loss: 1.1694 | Train Acc: 0.6078
Val Loss:   13.6727 | Val Acc:   0.0500

Trening zakończony w 0m 40s


epoch,▁▂▃▃▄▅▆▆▇█
total_time_seconds,▁
train_acc,▁▂▂▂▂▄▆▇▇█
train_loss,██▅▄▃▃▂▂▁▁
val_acc,▁▁▂▂▂▄▆▇▇█
val_loss,█▅▂▁▁▃▄▆▆█
epoch,10
total_time_seconds,40.18668
train_acc,0.60784
train_loss,1.16941
val_acc,0.05


# VGG13

## for scratch

W pracy tutaj była odpowiedź na pytania z zadania:

On a system equipped with
four NVIDIA Titan Black GPUs, training a single net took 2–3 weeks depending on the architecture.

ILSVRC-2012 dataset (which was used for ILSVRC 2012–2014 challenges). The dataset includes images of 1000 classes, and is split into three sets: training (1.3M
images), validation (50K images), and testing (100K images with held-out class labels). 

In [45]:
vgg13_from_scratch = models.vgg13(weights=None, num_classes=102).to(device)

total_params_vgg = sum(p.numel() for p in vgg13_from_scratch.parameters())
print(f"Liczba parametrów w VGG-13: {total_params_vgg:,}")

optimizer_vgg = optim.Adam(vgg13_from_scratch.parameters(), lr=0.01)

print("\nRozpoczynamy próbę trenowania VGG-13 od zera...")

history_vgg_scratch = train_model(
    model=vgg13_from_scratch, 
    train_loader=train_loader, 
    val_loader=val_loader, 
    criterion=criterion, 
    optimizer=optimizer_vgg, 
    epochs=10, 
    lr=0.01, 
    name="task0-vgg13-scratch", 
    architecture="vgg13_scratch"
)

Liczba parametrów w VGG-13: 129,368,742

Rozpoczynamy próbę trenowania VGG-13 od zera...


Epoka 1/10


Validation: 100%|██████████| 32/32 [00:10<00:00,  2.98it/s]


Train Loss: 13150043.4251 | Train Acc: 0.0039
Val Loss:   4.6257 | Val Acc:   0.0098

Epoka 2/10


Validation: 100%|██████████| 32/32 [00:10<00:00,  2.93it/s]


...

Validation: 100%|██████████| 32/32 [00:11<00:00,  2.79it/s]


Train Loss: 4.6300 | Train Acc: 0.0049
Val Loss:   4.6253 | Val Acc:   0.0098

Epoka 10/10


Validation: 100%|██████████| 32/32 [00:11<00:00,  2.69it/s]

Train Loss: 4.6602 | Train Acc: 0.0029
Val Loss:   4.6253 | Val Acc:   0.0098

Trening zakończony w 8m 25s


epoch,▁▂▃▃▄▅▆▆▇█
total_time_seconds,▁
train_acc,▃▆▆█▁▃▅▅▅▁
train_loss,█▁▁▁▁▁▁▁▁▁
val_acc,▁▁▁▁▁▁▁▁▁▁
val_loss,▄██▅▄▃▃▂▁▁
epoch,10
total_time_seconds,505.2371
train_acc,0.00294
train_loss,4.66015
val_acc,0.0098


Trenowanie głębokich sieci (jak VGG) od zera na małym zbiorze danych jest nieefektywne, zajmuje mnóstwo czasu i prowadzi do dobrych wyników. Sieć nie potrafi nauczyć się tak skomplikowanych cech z tak małej próbki. Dlatego Transfer Learning może być koniecznością.